# Run 4 — Résolution 384×384 avec batch effectif = 8

**Objectif** : Déconfondre l'effet de la résolution d'entrée vs. la taille de mini-batch.

L'expérience d'ablation `size=384 (bs=2)` avait obtenu un DSC de 0.769 — nettement inférieur à la baseline (0.854 à 256×256 bs=8). Ce résultat pourrait être dû au batch_size=2 (variance élevée du gradient) plutôt qu'à la résolution.

**Méthode** : Gradient accumulation (accum_steps=4, physical_batch=2) → effective_batch=8.

**Config** : Identique à la baseline d'ablation (scale=0.5) sauf `img_size=(384,384)`.

In [1]:
import os, time, json, gc
import glob as _glob
import numpy as np

# --- GPU config (Onyxia / cuDNN stability) ---
os.environ['TF_XLA_FLAGS'] = '--tf_xla_auto_jit=0'
os.environ['TF_CUDNN_USE_AUTOTUNE'] = '0'
os.environ['TF_USE_CUDNN_BATCHNORM_SPATIAL_PERSISTENT'] = '0'

import tensorflow as tf
tf.config.optimizer.set_experimental_options({'disable_meta_optimizer': True})

from tensorflow.keras import layers, models, backend as K
from sklearn.model_selection import StratifiedShuffleSplit

print(f"TensorFlow {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

2026-03-13 23:19:15.689191: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 1. Configuration

In [2]:
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# Paths (compatible Onyxia / Kaggle)
_kaggle = _glob.glob('/kaggle/input/*/dataset_ISIC')
if _kaggle:
    _base = _kaggle[0]
else:
    _base = 'dataset_ISIC'
IMAGES_DIR = os.path.join(_base, 'ISBI2016_ISIC_Part1_Training_Data')
MASKS_DIR  = os.path.join(_base, 'ISBI2016_ISIC_Part1_Training_GroundTruth')

# Hyperparamètres
IMG_SIZE       = (384, 384)
PHYS_BATCH     = 2          # ce qui tient en VRAM
ACCUM_STEPS    = 4          # gradient accumulation
EFF_BATCH      = PHYS_BATCH * ACCUM_STEPS  # = 8
EPOCHS         = 30
PATIENCE       = 8
LR             = 1e-4
SCALE          = 0.5        # même sous-ensemble que les ablations

SAVE_DIR = 'models'
os.makedirs(SAVE_DIR, exist_ok=True)
EXP_NAME = 'size=384_bs=8_gradaccum'

print(f"Résolution: {IMG_SIZE}, batch effectif: {EFF_BATCH} ({PHYS_BATCH}×{ACCUM_STEPS})")
print(f"Images: {IMAGES_DIR}")

Résolution: (384, 384), batch effectif: 8 (2×4)
Images: dataset_ISIC/ISBI2016_ISIC_Part1_Training_Data


## 2. Chargement des données

In [3]:
# ---------- helpers ----------
def load_image_mask(img_path, mask_path):
    img  = tf.image.decode_jpeg(tf.io.read_file(img_path), channels=3)
    mask = tf.image.decode_png(tf.io.read_file(mask_path), channels=1)
    img  = tf.image.convert_image_dtype(img, tf.float32)
    mask = tf.cast(mask > 127, tf.float32)
    return img, mask

def preprocess(img, mask, img_size):
    img  = tf.image.resize(img, img_size, method='bilinear')
    mask = tf.image.resize(mask, img_size, method='nearest')
    return img, mask

def augment_flip(img, mask):
    if tf.random.uniform(()) > 0.5:
        img  = tf.image.flip_left_right(img)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        img  = tf.image.flip_up_down(img)
        mask = tf.image.flip_up_down(mask)
    return img, mask

AUTOTUNE = tf.data.AUTOTUNE

def make_ds(img_files, mask_files, img_size, batch_size, augment_fn=None, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((img_files, mask_files))
    ds = ds.map(lambda i, m: load_image_mask(i, m), num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda i, m: preprocess(i, m, img_size), num_parallel_calls=AUTOTUNE)
    if augment_fn:
        ds = ds.map(augment_fn, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(len(img_files))
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

In [4]:
# ---------- split stratifié (identique aux ablations) ----------
def _compute_lesion_bin(mask_path):
    mask = tf.image.decode_png(tf.io.read_file(mask_path), channels=1)
    ratio = float(tf.reduce_mean(tf.cast(mask > 127, tf.float32)))
    if ratio < 0.01:   return 0, ratio
    elif ratio < 0.05: return 1, ratio
    elif ratio < 0.10: return 2, ratio
    else:              return 3, ratio

def _merge_rare_bins(bins, min_count=4):
    bins = bins.copy()
    unique, counts = np.unique(bins, return_counts=True)
    for b, c in zip(unique, counts):
        if c < min_count:
            bins[bins == b] = b - 1 if b > 0 else b + 1
            print(f'  [Warning] Bin {b} ({c} samples) merged')
    return bins

def split_paths(scale=0.5, val_ratio=0.2, seed=SEED):
    imgs  = sorted([os.path.join(IMAGES_DIR, f) for f in os.listdir(IMAGES_DIR) if f.endswith('.jpg')])
    masks = sorted([os.path.join(MASKS_DIR, f) for f in os.listdir(MASKS_DIR) if f.endswith('.png')])
    
    all_bins, all_ratios = [], []
    for mp in masks:
        b, r = _compute_lesion_bin(mp)
        all_bins.append(b); all_ratios.append(r)
    all_bins = np.array(all_bins)
    all_bins = _merge_rare_bins(all_bins)
    
    # Sous-échantillonnage stratifié
    all_idx = np.arange(len(imgs))
    if scale < 1.0:
        sss = StratifiedShuffleSplit(n_splits=1, train_size=scale, random_state=seed)
        keep_idx, _ = next(sss.split(all_idx, all_bins))
        imgs  = [imgs[i] for i in keep_idx]
        masks = [masks[i] for i in keep_idx]
        bins  = _merge_rare_bins(all_bins[keep_idx])
    else:
        bins = all_bins
    
    # Split train / val
    idx = np.arange(len(imgs))
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_ratio, random_state=seed)
    tr_idx, va_idx = next(sss2.split(idx, bins))
    
    to_list = lambda ii: ([imgs[i] for i in ii], [masks[i] for i in ii])
    return to_list(tr_idx), to_list(va_idx)

(train_img, train_mask), (val_img, val_mask) = split_paths(scale=SCALE)
print(f"Train: {len(train_img)}, Val: {len(val_img)}")

I0000 00:00:1773443962.170860    1602 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 722 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:86:00.0, compute capability: 7.5


Train: 360, Val: 90


In [5]:
train_ds = make_ds(train_img, train_mask, IMG_SIZE, PHYS_BATCH, augment_fn=augment_flip)
val_ds   = make_ds(val_img, val_mask, IMG_SIZE, PHYS_BATCH, augment_fn=None, shuffle=False)

## 3. Architecture U-Net

In [6]:
def conv_block(x, filters, activation='relu', use_batchnorm=True):
    for _ in range(2):
        x = layers.Conv2D(filters, 3, padding='same', kernel_initializer='he_normal')(x)
        if use_batchnorm:
            x = layers.BatchNormalization()(x)
        x = layers.Activation(activation)(x)
    return x

def build_unet(input_shape=(384,384,3), base_filters=64, depth=4,
               use_skip=True, dropout_rate=0.0, activation='relu',
               upsample='bilinear', use_batchnorm=True, pooling='max'):
    inputs = layers.Input(shape=input_shape)
    skips = []
    x = inputs
    pool_layer = layers.MaxPool2D if pooling == 'max' else layers.AveragePooling2D
    
    # Encoder
    for i in range(depth):
        x = conv_block(x, base_filters * (2**i), activation, use_batchnorm)
        skips.append(x)
        x = pool_layer((2,2))(x)
    
    # Bottleneck
    x = conv_block(x, base_filters * (2**depth), activation, use_batchnorm)
    if dropout_rate > 0:
        x = layers.Dropout(dropout_rate)(x)
    
    # Decoder
    for i in reversed(range(depth)):
        filters_i = base_filters * (2**i)
        if upsample == 'transpose':
            x = layers.Conv2DTranspose(filters_i, 2, strides=2, padding='same',
                                       kernel_initializer='he_normal')(x)
        else:
            x = layers.UpSampling2D((2,2))(x)
        if use_skip:
            x = layers.Concatenate()([x, skips[i]])
        x = conv_block(x, filters_i, activation, use_batchnorm)
    
    outputs = layers.Conv2D(1, 1, activation='sigmoid')(x)
    return models.Model(inputs, outputs)

model = build_unet(input_shape=(*IMG_SIZE, 3))
print(f"Params: {model.count_params():,}")

Params: 31,402,497


## 4. Métriques et loss

In [7]:
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f, y_pred_f = K.flatten(y_true), K.flatten(y_pred)
    inter = K.sum(y_true_f * y_pred_f)
    return (2. * inter + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def iou_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f, y_pred_f = K.flatten(y_true), K.flatten(y_pred)
    inter = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f) - inter
    return (inter + smooth) / (union + smooth)

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coefficient(y_true, y_pred)

def bce_dice_loss(y_true, y_pred):
    return tf.keras.losses.binary_crossentropy(y_true, y_pred) + dice_loss(y_true, y_pred)

loss_fn = bce_dice_loss
METRICS = [dice_coefficient, iou_coefficient, 'binary_accuracy']

## 5. Entraînement avec gradient accumulation

In [8]:
optimizer = tf.keras.optimizers.Adam(learning_rate=LR)

# Métriques de suivi
train_dice_metric = tf.keras.metrics.Mean(name='train_dice')
train_iou_metric  = tf.keras.metrics.Mean(name='train_iou')
train_loss_metric = tf.keras.metrics.Mean(name='train_loss')

@tf.function
def train_step_accum(images, masks, accum_grads):
    """Un micro-step : calcule les gradients et les accumule."""
    with tf.GradientTape() as tape:
        preds = model(images, training=True)
        loss  = loss_fn(masks, preds)
        loss  = tf.reduce_mean(loss)
    
    grads = tape.gradient(loss, model.trainable_variables)
    for i, g in enumerate(grads):
        if g is not None:
            accum_grads[i].assign_add(g)
    
    # Métriques sur ce micro-batch
    dice_val = dice_coefficient(masks, preds)
    iou_val  = iou_coefficient(masks, preds)
    train_dice_metric.update_state(dice_val)
    train_iou_metric.update_state(iou_val)
    train_loss_metric.update_state(loss)
    return loss

@tf.function
def apply_accum_gradients(accum_grads):
    """Applique les gradients moyennés et remet à zéro."""
    avg_grads = [g / ACCUM_STEPS for g in accum_grads]
    optimizer.apply_gradients(zip(avg_grads, model.trainable_variables))
    for g in accum_grads:
        g.assign(tf.zeros_like(g))

@tf.function
def val_step(images, masks):
    preds = model(images, training=False)
    loss  = tf.reduce_mean(loss_fn(masks, preds))
    dice_val = dice_coefficient(masks, preds)
    iou_val  = iou_coefficient(masks, preds)
    return loss, dice_val, iou_val

print("Fonctions de training compilées.")

Fonctions de training compilées.


In [9]:
# ---------- Boucle d'entraînement ----------
history = {
    'dice_coefficient': [], 'val_dice_coefficient': [],
    'iou_coefficient': [], 'val_iou_coefficient': [],
    'loss': [], 'val_loss': [],
    'binary_accuracy': [], 'val_binary_accuracy': []
}

best_val_dice = 0.0
best_val_iou  = 0.0
best_weights  = None
wait = 0

# Initialiser les accumulateurs de gradients
accum_grads = [tf.Variable(tf.zeros_like(v), trainable=False) 
               for v in model.trainable_variables]

t0 = time.time()

for epoch in range(EPOCHS):
    # --- Train ---
    train_dice_metric.reset_state()
    train_iou_metric.reset_state()
    train_loss_metric.reset_state()
    
    step_count = 0
    for images, masks in train_ds:
        train_step_accum(images, masks, accum_grads)
        step_count += 1
        
        if step_count % ACCUM_STEPS == 0:
            apply_accum_gradients(accum_grads)
    
    # Appliquer les gradients restants si le dernier batch n'est pas complet
    if step_count % ACCUM_STEPS != 0:
        apply_accum_gradients(accum_grads)
    
    tr_loss = float(train_loss_metric.result())
    tr_dice = float(train_dice_metric.result())
    tr_iou  = float(train_iou_metric.result())
    
    # --- Validation ---
    val_losses, val_dices, val_ious = [], [], []
    for images, masks in val_ds:
        vl, vd, vi = val_step(images, masks)
        val_losses.append(float(vl))
        val_dices.append(float(vd))
        val_ious.append(float(vi))
    
    va_loss = np.mean(val_losses)
    va_dice = np.mean(val_dices)
    va_iou  = np.mean(val_ious)
    
    # --- Historique ---
    history['loss'].append(tr_loss)
    history['dice_coefficient'].append(tr_dice)
    history['iou_coefficient'].append(tr_iou)
    history['binary_accuracy'].append(0.0)  # non suivi dans cette boucle
    history['val_loss'].append(va_loss)
    history['val_dice_coefficient'].append(va_dice)
    history['val_iou_coefficient'].append(va_iou)
    history['val_binary_accuracy'].append(0.0)
    
    # --- Early stopping sur val_dice ---
    if va_dice > best_val_dice:
        best_val_dice = va_dice
        best_val_iou  = va_iou
        best_weights  = model.get_weights()
        wait = 0
        marker = ' *'
    else:
        wait += 1
        marker = ''
    
    elapsed = time.time() - t0
    print(f"Epoch {epoch+1:2d}/{EPOCHS} — "
          f"loss: {tr_loss:.4f}  dice: {tr_dice:.4f}  iou: {tr_iou:.4f} | "
          f"val_loss: {va_loss:.4f}  val_dice: {va_dice:.4f}  val_iou: {va_iou:.4f} "
          f"[{elapsed:.0f}s]{marker}")
    
    if wait >= PATIENCE:
        print(f"\nEarly stopping à l'epoch {epoch+1} (patience={PATIENCE})")
        break

# Restaurer les meilleurs poids
if best_weights is not None:
    model.set_weights(best_weights)

total_time = time.time() - t0
stopped_epoch = len(history['loss'])
print(f"\nTerminé en {total_time:.1f}s ({stopped_epoch} epochs)")
print(f"Meilleur val_dice: {best_val_dice:.4f}, val_iou: {best_val_iou:.4f}")

2026-03-13 23:19:44.123192: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 90501
2026-03-13 23:19:45.496128: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 88.00MiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
W0000 00:00:1773443985.496423    1890 gpu_utils.cc:68] Failed to allocate memory for convolution redzone checking; skipping this check. This is benign and only means that we won't check cudnn for out-of-bounds reads and writes. This message will only be printed once.
2026-03-13 23:19:45.586851: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 52.00MiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memor

ResourceExhaustedError: Graph execution error:

Detected at node functional_1/conv2d_3_1/convolution defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/opt/python/lib/python3.13/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/opt/python/lib/python3.13/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/opt/python/lib/python3.13/site-packages/ipykernel/kernelapp.py", line 758, in start

  File "/opt/python/lib/python3.13/site-packages/tornado/platform/asyncio.py", line 211, in start

  File "/opt/python/lib/python3.13/asyncio/base_events.py", line 683, in run_forever

  File "/opt/python/lib/python3.13/asyncio/base_events.py", line 2050, in _run_once

  File "/opt/python/lib/python3.13/asyncio/events.py", line 89, in _run

  File "/opt/python/lib/python3.13/site-packages/ipykernel/kernelbase.py", line 621, in shell_main

  File "/o

## 6. Sauvegarde des résultats

In [ ]:
def _safe_name(name):
    return name.replace(' ', '_').replace('/', '-').replace('(', '').replace(')', '')

res = dict(
    name=EXP_NAME,
    best_val_dice=best_val_dice,
    best_val_iou=best_val_iou,
    params=model.count_params(),
    time_s=round(total_time, 1),
    history=history,
    stopped_epoch=stopped_epoch,
    config=dict(
        img_size=list(IMG_SIZE),
        physical_batch=PHYS_BATCH,
        accum_steps=ACCUM_STEPS,
        effective_batch=EFF_BATCH,
        lr=LR,
        scale=SCALE,
        loss='bce_dice',
        augmentation='flip',
        base_filters=64,
        depth=4,
        use_skip=True,
        batchnorm=True,
        pooling='max',
        dropout=0.0
    )
)

# Sauvegarder JSON
json_path = os.path.join(SAVE_DIR, f'{_safe_name(EXP_NAME)}.json')
with open(json_path, 'w') as f:
    json.dump(res, f)
print(f"Résultats sauvegardés: {json_path}")

# Sauvegarder modèle
model_path = os.path.join(SAVE_DIR, f'{_safe_name(EXP_NAME)}.keras')
model.save(model_path)
print(f"Modèle sauvegardé: {model_path}")

## 7. Comparaison avec les résultats précédents

In [ ]:
import pandas as pd

# Charger les résultats de référence
comparisons = {
    'Baseline (256×256, bs=8)': 'baseline_f64-d4-skip-bce+dice',
    '384×384, bs=2 (ancien)': 'size=384_bs=2',
    '384×384, bs=8 (grad accum)': EXP_NAME,
}

rows = []
for label, name in comparisons.items():
    path = os.path.join(SAVE_DIR, f'{_safe_name(name)}.json')
    if os.path.exists(path):
        with open(path) as f:
            r = json.load(f)
        rows.append({
            'Configuration': label,
            'DSC': f"{r['best_val_dice']:.4f}",
            'IoU': f"{r['best_val_iou']:.4f}",
            'Epochs': r['stopped_epoch'],
            'Temps (s)': r['time_s'],
        })
    else:
        print(f"[!] Fichier non trouvé: {path}")

df = pd.DataFrame(rows)
print("\n=== Comparaison ===")
print(df.to_string(index=False))

# Interpréter
if len(rows) >= 3:
    old_384 = float(rows[1]['DSC'])
    new_384 = float(rows[2]['DSC'])
    baseline = float(rows[0]['DSC'])
    print(f"\nΔ DSC (bs=8 vs bs=2 à 384): {new_384 - old_384:+.4f}")
    print(f"Δ DSC (384 bs=8 vs baseline 256): {new_384 - baseline:+.4f}")
    if new_384 > old_384 + 0.03:
        print("→ La dégradation était largement due au batch size, pas à la résolution.")
    elif new_384 > old_384 + 0.01:
        print("→ Le batch size explique partiellement la dégradation. Effet combiné résolution + batch.")
    else:
        print("→ La dégradation est principalement due à la résolution, pas au batch size.")

In [ ]:
# Courbes d'entraînement
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['dice_coefficient'], label='Train Dice')
axes[0].plot(history['val_dice_coefficient'], label='Val Dice')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Dice')
axes[0].legend(); axes[0].set_title('Dice coefficient')
axes[0].axhline(y=best_val_dice, color='r', linestyle='--', alpha=0.5, label=f'Best: {best_val_dice:.4f}')
axes[0].legend()

axes[1].plot(history['loss'], label='Train Loss')
axes[1].plot(history['val_loss'], label='Val Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].set_title('Loss')

fig.suptitle(f'384×384 — batch effectif = {EFF_BATCH} (gradient accumulation)', fontsize=13)
plt.tight_layout()
plt.savefig('figures/fig_384_gradaccum_curves.pdf', bbox_inches='tight')
plt.show()